In [2]:
from fastembed import TextEmbedding
import numpy as np

# this downloads the model on first run (~50MB)
embedding_model = TextEmbedding("BAAI/bge-small-en-v1.5")

# test embedding
texts = ["How do I join the course?"]
embeddings = list(embedding_model.embed(texts))

embedding = embeddings[0]
print(f"Embedding type: {type(embedding)}")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding type: <class 'numpy.ndarray'>
Embedding dimension: 384
First 5 values: [ 0.00515081 -0.05800219 -0.00891525 -0.06819104  0.01400968]


In [3]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sentences = [
    "How do I join the course?",           # query
    "Can I still enroll in the class?",    # similar meaning
    "What is the best pizza topping?",     # completely unrelated
]

embeddings = list(embedding_model.embed(sentences))

query = embeddings[0]
similar = embeddings[1]
unrelated = embeddings[2]

print(f"Query vs Similar:   {cosine_similarity(query, similar):.4f}")
print(f"Query vs Unrelated: {cosine_similarity(query, unrelated):.4f}")

Query vs Similar:   0.7467
Query vs Unrelated: 0.3651


In [4]:
# reuse our documents from before
documents = [
    {"id": "1", "course": "data-engineering-zoomcamp", "section": "General", "question": "Can I still join the course after the start date?", "answer": "Yes, you can still join. There is no penalty for joining late."},
    {"id": "2", "course": "data-engineering-zoomcamp", "section": "General", "question": "What are the prerequisites for this course?", "answer": "You need Python, SQL, and command line experience."},
    {"id": "3", "course": "data-engineering-zoomcamp", "section": "General", "question": "How do I get a certificate?", "answer": "Complete the capstone project and get it peer reviewed."},
    {"id": "4", "course": "data-engineering-zoomcamp", "section": "Docker", "question": "How do I run a container in Docker?", "answer": "Use docker run <image-name>. Add -it for interactive mode."},
    {"id": "5", "course": "llm-zoomcamp", "section": "General", "question": "What is RAG?", "answer": "RAG stands for Retrieval Augmented Generation. It combines search with LLM generation."},
    {"id": "6", "course": "llm-zoomcamp", "section": "General", "question": "What LLM providers can I use?", "answer": "You can use OpenAI, Groq, Google Gemini, or any OpenAI-compatible provider."},
]

# embed all documents (question + answer combined)
print("Embedding documents...")
texts = [f"{doc['question']} {doc['answer']}" for doc in documents]
doc_embeddings = list(embedding_model.embed(texts))
doc_embeddings = np.array(doc_embeddings)

print(f"Embeddings shape: {doc_embeddings.shape}")

Embedding documents...
Embeddings shape: (6, 384)


In [5]:
def vector_search(query, top_k=3):
    # embed the query
    query_embedding = list(embedding_model.embed([query]))[0]
    
    # compute cosine similarity with all documents
    similarities = []
    for i, doc_emb in enumerate(doc_embeddings):
        score = cosine_similarity(query_embedding, doc_emb)
        similarities.append((score, i))
    
    # sort by similarity descending
    similarities.sort(reverse=True)
    
    # return top k documents
    results = []
    for score, idx in similarities[:top_k]:
        doc = documents[idx].copy()
        doc['score'] = round(score, 4)
        results.append(doc)
    
    return results

# test it
results = vector_search("I want to enroll, is it too late?")
for r in results:
    print(f"Score: {r['score']} | {r['question']}")

Score: 0.7436000108718872 | Can I still join the course after the start date?
Score: 0.6208999752998352 | What are the prerequisites for this course?
Score: 0.5620999932289124 | How do I get a certificate?


In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

# in-memory mode - no server needed
qdrant = QdrantClient(":memory:")

# create a collection
qdrant.create_collection(
    collection_name="faq",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

print("Collection created ")

Collection created 


In [8]:
points = []
for i, (doc, embedding) in enumerate(zip(documents, doc_embeddings)):
    points.append(PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload=doc
    ))

qdrant.upsert(collection_name="faq", points=points)
print(f"Indexed {len(points)} documents into Qdrant ")

Indexed 6 documents into Qdrant 


In [9]:
def qdrant_search(query, top_k=3):
    query_embedding = list(embedding_model.embed([query]))[0]
    
    results = qdrant.query_points(
        collection_name="faq",
        query=query_embedding.tolist(),
        limit=top_k
    )
    
    return results.points

results = qdrant_search("I want to enroll, is it too late?")
for r in results:
    print(f"Score: {r.score:.4f} | {r.payload['question']}")

Score: 0.7436 | Can I still join the course after the start date?
Score: 0.6209 | What are the prerequisites for this course?
Score: 0.5621 | How do I get a certificate?


In [10]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def build_prompt(question, search_results):
    context = ""
    for r in search_results:
        doc = r.payload
        context += f"Q: {doc['question']}\nA: {doc['answer']}\n\n"
    
    return f"""You're a course assistant. Answer based on CONTEXT only.
If not in context, say "I don't know."

QUESTION: {question}

CONTEXT:
{context}""".strip()

def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def vector_rag(question):
    search_results = qdrant_search(question, top_k=3)
    prompt = build_prompt(question, search_results)
    return llm(prompt)

# test
answer = vector_rag("I want to enroll, is it too late?")
print(answer)

No, it's not too late to enroll.
